<a href="https://colab.research.google.com/github/basmalaazab/HandTalk/blob/main/handtalk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip uninstall -y protobuf mediapipe tensorflow
!pip install protobuf==4.25.3
!pip install mediapipe==0.10.14

Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6
Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 7.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf-tf 2.20.0 requires tensorflow==2.20.0, which is not installed.
google-cloud-datastore 2.25.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
google-cloud-resource-manager 1.18.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
google-cloud-translate 3.27.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.3 which i

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 15.3 MB/s eta 0:00:00


In [1]:
# Import Libraries
import os
import zipfile
import cv2
import mediapipe as mp
import pandas as pd

In [2]:
# Configuration
ZIP_FILE = "/content/archive_4.zip"
EXTRACT_PATH = "/content/dataset"
OUTPUT_FILE = "/content/processed_dataset.csv"

In [3]:
# Extract Dataset
with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extracted successfully.")

Dataset extracted successfully.


In [4]:
# Locate Dataset Folder
folders = os.listdir(EXTRACT_PATH)
if len(folders) == 1:
    DATA_DIR = os.path.join(EXTRACT_PATH, folders[0])
else:
    DATA_DIR = EXTRACT_PATH
print("Dataset Folder:", DATA_DIR)

Dataset Folder: /content/dataset/data


In [6]:
# Initialize MediaPipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
)

In [7]:
# Create Empty Lists
data = []
labels = []

In [8]:
# Read Dataset
for label in sorted(os.listdir(DATA_DIR)):
    class_folder = os.path.join(DATA_DIR, label)
    if not os.path.isdir(class_folder):
        continue
    print(f"Processing Class: {label}")
    for image_name in os.listdir(class_folder):
        image_path = os.path.join(class_folder, image_name)
        image = cv2.imread(image_path)
        if image is None:
            continue

Processing Class: 1
Processing Class: 2
Processing Class: 3
Processing Class: 4
Processing Class: 5
Processing Class: 6
Processing Class: 7
Processing Class: 8
Processing Class: 9
Processing Class: A
Processing Class: B
Processing Class: C
Processing Class: D
Processing Class: E
Processing Class: F
Processing Class: G
Processing Class: H
Processing Class: I
Processing Class: J
Processing Class: K
Processing Class: L
Processing Class: M
Processing Class: N
Processing Class: O
Processing Class: P
Processing Class: Q
Processing Class: R
Processing Class: S
Processing Class: T
Processing Class: U
Processing Class: V
Processing Class: W
Processing Class: X
Processing Class: Y
Processing Class: Z


In [10]:
# Convert image to RGB
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [13]:
# Detect hand
results = hands.process(image_rgb)

if results.multi_hand_landmarks:
    hand_landmarks = results.multi_hand_landmarks[0]
    x_coordinates = []
    y_coordinates = []
    for landmark in hand_landmarks.landmark:
        x_coordinates.append(landmark.x)
        y_coordinates.append(landmark.y)
    min_x = min(x_coordinates)
    min_y = min(y_coordinates)
    features = []
    for landmark in hand_landmarks.landmark:
        features.append(landmark.x - min_x)
        features.append(landmark.y - min_y)
    if len(features) == 42:
        data.append(features)
        labels.append(label)

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [14]:
# Create DataFrame
feature_names = []
for i in range(21):
    feature_names.append(f"x{i+1}")
    feature_names.append(f"y{i+1}")
df = pd.DataFrame(data, columns=feature_names)
df["label"] = labels

In [15]:
# Save Processed Dataset
df.to_csv(OUTPUT_FILE, index=False)

print("\n====================================")
print("Preprocessing Completed Successfully")
print("Dataset Shape :", df.shape)
print("Saved File :", OUTPUT_FILE)
print("====================================")


Preprocessing Completed Successfully
Dataset Shape : (2, 43)
Saved File : /content/processed_dataset.csv
